In [1]:
import numpy as np
import pandas as pd
import mlflow.sklearn
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [2]:
#dataset ingestion
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

print(housing.keys())


dict_keys(['data', 'target', 'frame', 'target_names', 'feature_names', 'DESCR'])


In [3]:
#prepapring the dataset
df = pd.DataFrame(housing.data,columns=housing.feature_names)

In [4]:
df.head(2)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.02381,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.97188,2401.0,2.109842,37.86,-122.22


In [5]:
#Extractting target column
df['price'] = housing.target

In [6]:
df.columns

Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude', 'price'],
      dtype='object')

In [7]:
#train test split Model Hyperparameter Tuning

from urllib.parse import urlparse


X = df.drop("price", axis=1)
y = df["price"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
def hyperparameter_tuning(X_train, y_train,param_dist):
    rf = RandomForestRegressor()
    random_search = RandomizedSearchCV(
        rf, 
        param_distributions=param_dist, 
        n_iter=10, 
        cv=5, 
        verbose=2,
        scoring='neg_mean_squared_error', 
        random_state=42)
    random_search.fit(X_train, y_train)
    return random_search
   

In [13]:
from mlflow.models import infer_signature
from urllib.parse import urlparse

signature1 = infer_signature(X_train, y_train)

# Define the Hyperparameters
params = {
    "n_estimators": [100, 200, 300, 400, 500],       
    "max_depth": [4, 6, 8, 10, None],                 
    "min_samples_split": [2, 5, 10],                  
    "min_samples_leaf": [1, 2, 4],                    
    "random_state": [42]                              
}

# Set tracking URI BEFORE starting the run
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# Start MLflow experiment
with mlflow.start_run():
    random_search = hyperparameter_tuning(X_train, y_train, params)

    # Get the best model
    best_model = random_search.best_estimator_

    # Evaluate the model
    y_pred = best_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)

    mlflow.log_param("best_n_estimators",    random_search.best_params_["n_estimators"])
    mlflow.log_param("best_max_depth",       random_search.best_params_["max_depth"])
    mlflow.log_param("best_min_samples_split", random_search.best_params_["min_samples_split"])
    mlflow.log_param("best_min_samples_leaf",  random_search.best_params_["min_samples_leaf"])
    mlflow.log_param("best_random_state",    random_search.best_params_["random_state"])
    mlflow.log_metric("mse", mse)

    
    tracking_uri_type_store = urlparse(mlflow.get_tracking_uri()).scheme

    if tracking_uri_type_store != "file":
        mlflow.sklearn.log_model(best_model, "model",
                                 registered_model_name="Best Random Search Model",
                                 signature=signature1)
    else:
        mlflow.sklearn.log_model(best_model, "model", signature=signature1)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=5, n_estimators=500, random_state=42; total time=   9.4s
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=5, n_estimators=500, random_state=42; total time=   9.0s
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=5, n_estimators=500, random_state=42; total time=   9.2s
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=5, n_estimators=500, random_state=42; total time=   9.1s
[CV] END max_depth=4, min_samples_leaf=1, min_samples_split=5, n_estimators=500, random_state=42; total time=   9.1s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=500, random_state=42; total time=  30.5s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=500, random_state=42; total time=  32.1s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=500, random_state=42; total time=  

2026/03/08 22:24:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/08 22:24:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Best Random Search Model'.
2026/03/08 22:28:13 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Best Random Search Model, version 1
Created version '1' of model 'Best Random Search Model'.


🏃 View run kindly-ram-840 at: http://127.0.0.1:5000/#/experiments/0/runs/d9b83e11509242eea971bd4d7c1b90c6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
